# Clinic Administrator Demo

**Astra** is an AI administrator for an aesthetic medicine clinic: answers about treatments, doctor schedules, client registration, booking and cancellation — grounded in the clinic **live database**, not model memory.

The notebook **demonstrates** one demo client's journey. Markdown cells describe what each step shows; code cells run the production application from the private repository.

**Scenario demonstrated:** visitor asks about Botox -> free slot lookup -> registration -> booking -> calendar check -> cancellation.

**Language model in production (Telegram):** after name/phone onboarding, **each client message** is handled by the dialog agent. One visible reply can involve several internal model calls: input safety check, tool selection, optional **text-to-SQL** when the procedure catalog is searched, reply drafting, output safety check. Onboarding and registration field prompts (birth date, email) use **fixed templates**, not the model.

**What this notebook runs:** section **3** is the only step that invokes the full dialog agent (one procedure-consultation turn). Sections **4–8** call the **same database services** the agent's tools use, so slot blocking and appointment rows are shown as structured output without simulating every chat turn.

**Not demonstrated here:** Telegram UI, Redis queue and background worker, per-message safety classifiers (disabled in section 3).

**Prerequisites:** production code should be imported in the Jupyter notebook (can't be re-run without it); PostgreSQL with clinic data, local LLM server, configuration in place.

In [ ]:
from __future__ import annotations

import asyncio
import json
from datetime import datetime, timedelta

import nest_asyncio
from sqlalchemy import text

nest_asyncio.apply()

import config
from db import dispose_engine, get_engine
from dialog_agent import DialogAgent
from dialog_agent.user_session import UserSessionStore
from guardrails.schemas import InputGuardrailVerdict, OutputGuardrailVerdict
from tools.book_appointment.service import BookAppointmentService
from tools.cancel_appointment.service import CancelAppointmentService
from tools.client_register.service import ClientRegisterService
from tools.get_appointment.service import GetAppointmentService
from tools.procedure_profile import PROCEDURE_PROFILES_CTX_KEY
from tools.schedule_check.service import ScheduleCheckService

In [ ]:
def run_async(coro):
    """Run async code inside Jupyter."""
    return asyncio.get_event_loop().run_until_complete(coro)

In [ ]:
# Demo user — DEMO_USER_ID is the session key (same role as telegram user id in worker).
DEMO_USER_ID = 9900001
DEMO_FULL_NAME = "Daniella Shlomi"
DEMO_PHONE = "+972501234567"
DEMO_BIRTH_DATE = "15.05.1990"
DEMO_EMAIL = "demo.client@example.com"
DEMO_PROCEDURE_QUERY = (
    "Hi! I'm thinking about botox for my forehead lines — "
    "how much would it cost and roughly how long is the appointment?"
)

## 1. Clinic data *(notebook preflight only)*

Not part of the live client flow. Shown here so the demo can confirm PostgreSQL is reachable and seeded before the scenario starts.

Demonstrated: table counts (procedures, doctors, clients, appointments) and a sample of treatments with prices in NIS. In production, the catalog is queried when the client asks — there is no separate check before each chat.

**Shown in output:** non-zero procedure count and a short price list.

In [ ]:
# Define snapshot reader once; same schema the worker uses at runtime.
async def print_database_snapshot() -> None:
    """Print table counts and a sample of cheapest procedures."""
    table_queries = {
        "procedures": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.procedures",
        "doctors": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.doctors",
        "clients": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.clients",
        "appointments": f"SELECT COUNT(*) FROM {config.DB_SCHEMA}.appointments",
    }
    async with get_engine().connect() as conn:
        for table_name, query in table_queries.items():
            row_count = (await conn.execute(text(query))).scalar_one()
            print(f"{table_name}: {row_count}")

        sample_result = await conn.execute(
            text(
                f"""
                SELECT doctor, procedure, cost, anestesia
                FROM {config.DB_SCHEMA}.procedures
                ORDER BY cost ASC
                LIMIT 5
                """
            )
        )
        print("\nSample procedures (prices in NIS):")
        for procedure_row in sample_result.fetchall():
            print(
                f"  • {procedure_row.procedure} — Dr {procedure_row.doctor}, "
                f"{procedure_row.cost} NIS, anesthesia: {procedure_row.anestesia}"
            )

In [ ]:
# Sanity check: DB reachable and seeded before simulating the client journey.
run_async(print_database_snapshot())

procedures: 20
doctors: 5
clients: 2
appointments: 2

Sample procedures (prices in NIS):
  • Lip augmentation consultation — Dr Smith E.M., 200 NIS, anesthesia: none
  • LED light therapy — Dr Davis L.Yu., 250 NIS, anesthesia: none
  • Gummy smile correction — Dr Johnson P.N., 300 NIS, anesthesia: none
  • Light chemical peel — Dr Smith E.M., 350 NIS, anesthesia: none
  • Laser hair removal (underarms) — Dr Williams A.A., 400 NIS, anesthesia: none


## 2. Onboarding

In production, identity is collected **before** the dialog agent runs (name and phone — same as at reception). The worker handles this path without calling the language model.

**Demonstrated:**
- **Step 1 (name):** validation of full name; fixed prompt asking for phone.
- **Step 2 (phone):** validation; fixed welcome message offering procedure info or booking.

After onboarding, the verified profile (name, phone, registration flag) is injected into every subsequent agent turn.

**Shown in output:** lines prefixed **Bot:**; after step 2, welcome text and the verified profile object.

In [ ]:
# Same as /start in beauty_worker: clear in-memory state for a fresh demo run.
session_store = UserSessionStore()
session_store.reset(DEMO_USER_ID)

UserSession(full_name=None, phone=None, onboarding_step=<OnboardingStep.AWAITING_NAME: 'awaiting_name'>, registration_step=None, birth_date=None, email=None, birth_date_attempts=0, email_attempts=0, pending_registration_prompt=False, is_registered=False, client_id=None)

In [ ]:
# Onboarding step 1: validate full name (first + last) before agent access.
name_step = session_store.handle_onboarding(DEMO_USER_ID, DEMO_FULL_NAME)
print("Bot:", name_step.reply or "(name accepted)")

Bot: Thank you! Please enter your phone number (e.g. +79991234567 or +972501234567).


In [ ]:
# Onboarding step 2: phone completes verified profile injected into every agent turn.
phone_step = session_store.handle_onboarding(DEMO_USER_ID, DEMO_PHONE)
print("Bot:", phone_step.reply)

user_session = session_store.get(DEMO_USER_ID)
verified_profile = user_session.to_profile()
print("\nVerified profile:", verified_profile)

Bot: Thank you! How can I help you today — procedure information or booking an appointment?

Verified profile: UserProfile(full_name='Daniella Shlomi', phone='+972501234567', is_registered=False, client_id=None)


## 3. Procedure consultation

The client question is in plain language (example: Botox for forehead lines — price and duration). This section runs a **full dialog-agent turn** — the same entry point the Telegram worker uses after onboarding.

**Demonstrated flow (one client-visible turn, multiple model calls):**
1. Input safety check (bypassed in this notebook; active in production).
2. The agent model selects the procedure-search tool and passes the client question.
3. Inside that tool, a **separate model call** converts the question to SQL and runs it against the procedure catalog.
4. Matching rows (doctor, price, duration, anesthesia) are stored in session context for later booking.
5. The agent model drafts the **client-facing reply** from tool results and dialog history.

In production the same client may trigger further agent turns (schedule, registration, booking) across later messages; here only consultation is shown.

**Shown in output:**
- Client question text.
- **Astra:** summary with procedure name, doctor, approximate duration, price in NIS, anesthesia type (mandatory / optional / none).
- Internal procedure profile retained for sections 4–6 (same data the booking step reads from session context).

In [ ]:
dialog_agent = DialogAgent(session_store)


async def _allow_input(*args, **kwargs) -> InputGuardrailVerdict:
    return InputGuardrailVerdict(allowed=True)


async def _allow_output(*args, **kwargs) -> OutputGuardrailVerdict:
    return OutputGuardrailVerdict(allowed=True)


dialog_agent._guardrails.check_input = _allow_input
dialog_agent._guardrails.check_output = _allow_output

run_async(dialog_agent.reset(DEMO_USER_ID))


async def ask_agent(user_text: str) -> str:
    current_session = session_store.get(DEMO_USER_ID)
    return await dialog_agent.handle(
        user_text,
        DEMO_USER_ID,
        profile=current_session.to_profile(),
    )


async def get_procedure_profiles_from_context() -> list[dict]:
    session = dialog_agent._session(DEMO_USER_ID)
    stored = await session.context.store.get(PROCEDURE_PROFILES_CTX_KEY, default={})
    return list(stored.values())

In [ ]:
# Client question in natural language — NOT SQL. Agent calls rag internally.
client_reply = run_async(ask_agent(DEMO_PROCEDURE_QUERY))
procedure_profiles = run_async(get_procedure_profiles_from_context())

print("Client question:", DEMO_PROCEDURE_QUERY)
print("\n--- Astra ---")
print(client_reply)
print("\n--- rag stored in context (internal, for booking demo) ---")
print(f"{len(procedure_profiles)} profile(s)")
for profile in procedure_profiles:
    print(profile)

Client question: Hi! I'm thinking about botox for my forehead lines — how much would it cost and roughly how long is the appointment?

--- Astra ---
Hello Daniella! 

For Botox treatment for forehead lines, the appointment costs **650 NIS** and takes approximately **one hour**. 

The procedure is performed by Dr. Smith E.M. and requires mandatory anesthesia, which is included in the total cost of **1,050 NIS**.

Would you like to book an appointment for this treatment? If so, I can check available times with Dr. Smith.

--- rag stored in context (internal, for booking demo) ---
1 profile(s)
{'procedure_id': 1, 'doctor': 'Smith E.M.', 'procedure': 'Botox (forehead)', 'cost': 650, 'anestesia': 'mandatory', 'if_anestesia_cost': 1050, 'duration': 60}


In [ ]:
# Booking steps below need procedure_id from rag context (same as book_appointment tool).
if not procedure_profiles:
    raise RuntimeError(
        "No procedure profiles in context after agent turn. "
        "Re-run section 3 setup cell (dialog_agent.reset) then the ask_agent cell."
    )

selected_procedure = procedure_profiles[0]
print("Selected for booking demo:", selected_procedure)

Selected for booking demo: {'procedure_id': 1, 'doctor': 'Smith E.M.', 'procedure': 'Botox (forehead)', 'cost': 650, 'anestesia': 'mandatory', 'if_anestesia_cost': 1050, 'duration': 60}


## 4. Doctor schedule

A concrete appointment time is required. Each specialist has a **day calendar** in 20-minute steps; each step is marked free or busy.

**Demonstrated here:** direct schedule lookup (same database logic the agent uses when checking availability).

**In Telegram:** the client would ask in natural language; the agent would look up slots and phrase available times in a reply. The notebook skips that dialogue to show raw slot data.

Demonstrated for the doctor from section 3 and **tomorrow's date**.

**Shown in output:**
- Full slot list for that day (`free` / `busy`).
- Summary: date, sample of free times, and the first free slot chosen for the booking demo.

In [ ]:
selected_doctor = selected_procedure["doctor"]
booking_date = (datetime.now() + timedelta(days=1)).strftime("%d.%m.%Y")

schedule_service = ScheduleCheckService()
schedule_result = run_async(
    schedule_service.check_schedule(selected_doctor, booking_date)
)
print(json.dumps(schedule_result, indent=2, default=str))

{
  "success": true,
  "doctor": "Smith E.M.",
  "date": "27.06.2026",
  "total_slots": 24,
  "free_slots": 24,
  "busy_slots": 0,
  "slots": [
    {
      "time": "09:00",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "09:20",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "09:40",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "10:00",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "10:20",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "10:40",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "11:00",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "12:20",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "12:40",
      "status": "free",
      "appointment_id": null
    },
    {
      "time": "13:00",
      "status": "free",
  

In [ ]:
# Pick first free slot for the booking demo.
free_slot_times = [
    slot["time"]
    for slot in schedule_result.get("slots", [])
    if slot["status"] == "free"
]
booking_time = free_slot_times[0] if free_slot_times else "09:00"

print(f"Doctor: {selected_doctor}")
print(f"Date: {booking_date}")
print(
    f"Free slots: {', '.join(free_slot_times[:8]) or '(none — using demo fallback 09:00)'}"
)
print(f"Selected time for demo: {booking_time}")

Doctor: Smith E.M.
Date: 27.06.2026
Free slots: 09:00, 09:20, 09:40, 10:00, 10:20, 10:40, 11:00, 12:20
Selected time for demo: 09:00


## 5. Clinic registration

Booking requires a **client profile** in the clinic database (birth date and email). Returning clients already in the database skip this block.

**Demonstrated for a new client:**
1. Lookup by phone — already registered or not.
2. If not registered — fixed prompts for **birth date**, then **email** (values validated and written to the database; birth date and email are **not** passed into the agent as free text during collection).
3. Profile created and linked to the demo account.

**In Telegram:** registration usually starts when the agent initiates it after explicit client consent; the worker sends the fixed prompts. After successful registration the worker may run **another agent turn** with a system hint so the model can continue booking — that follow-up turn is **not** shown here.

**Shown in output:** registration status, bot prompts, success message when complete.

In [ ]:
register_service = ClientRegisterService()

existing_client = run_async(
    register_service.get_client_by_telegram_id(DEMO_USER_ID)
)
if existing_client is None:
    phone_lookup = run_async(register_service.check_client_exists(DEMO_PHONE))
    if phone_lookup.get("exists"):
        existing_client = phone_lookup["client"]

print("Existing client:", existing_client)

Existing client: None


In [ ]:
if existing_client:
    demo_client_id = existing_client["id"]
    user_session.apply_client_record(existing_client)
    print(
        f"Already registered: {existing_client['full_name']} "
        f"(internal client_id={demo_client_id})"
    )
else:
    session_store.start_registration(DEMO_USER_ID, DEMO_FULL_NAME, DEMO_PHONE)
    print("Bot:", session_store.ASK_BIRTH_DATE)
    run_async(session_store.handle_registration(DEMO_USER_ID, DEMO_BIRTH_DATE))
    print("Bot:", session_store.ASK_EMAIL)
    registration_result = run_async(
        session_store.handle_registration(DEMO_USER_ID, DEMO_EMAIL)
    )
    print("Bot:", registration_result.user_reply)
    demo_client_id = user_session.client_id
    print(f"Registered with internal client_id={demo_client_id}")

Bot: To complete your clinic registration, please enter your date of birth (e.g. 15.05.1990 or 1990-05-15).
Bot: Please enter your email address (e.g. name@example.com).
Bot: Your clinic profile has been registered successfully.
Registered with internal client_id=11


## 6. Book appointment

**Demonstrated here:** direct booking (same database logic the agent uses): consecutive slots blocked for procedure length (e.g. ~60 minutes -> three 20-minute cells), appointment row created with the correct **total price** when anesthesia is mandatory.

**In Telegram:** the agent confirms date, time, and anesthesia choice in dialogue, then performs the booking and summarizes success in natural language.

On repeated runs with the same client, date, time, and procedure, an existing visit is **updated** rather than duplicated — `"updated": true` may appear in the JSON.

**Shown in output:** `"success": true` and a created-or-updated confirmation message.

In [ ]:
client_wants_anesthesia = selected_procedure.get("anestesia") == "mandatory"
appointment_duration = int(selected_procedure["duration"])

book_service = BookAppointmentService()
book_result = run_async(
    book_service.book(
        client_id=demo_client_id,
        procedure_id=int(selected_procedure["procedure_id"]),
        date=booking_date,
        time=booking_time,
        client_want_anestesia=client_wants_anesthesia,
        duration=appointment_duration,
    )
)
print(json.dumps(book_result, indent=2, default=str))

{
  "success": true,
  "appointment_id": 10,
  "message": "Appointment created successfully",
  "updated": false
}


## 7. Upcoming visits

**Demonstrated here:** direct appointment lookup (same database logic the agent uses when listing visits).

**In Telegram:** the client would ask in natural language; the agent would fetch the list and format it in a reply.

**Shown in output:** procedure name, doctor, date, time, duration, total cost in NIS — matching the booking from section 6.

In [ ]:
appointment_service = GetAppointmentService()
appointments_response = run_async(
    appointment_service.get_appointment(demo_client_id)
)

print(json.dumps(appointments_response, indent=2, default=str))

{
  "success": true,
  "appointments": [
    {
      "id": 10,
      "client_id": 11,
      "date": "27.06.2026",
      "time": "09:00",
      "procedure_id": 1,
      "duration": 60,
      "cost": 1050,
      "anestesia": "mandatory",
      "anestesia_used": "yes",
      "client_name": "Daniella Shlomi",
      "procedure_name": "Botox (forehead)",
      "doctor": "Smith E.M."
    }
  ],
  "count": 1
}


## 8. Cancel appointment

**Demonstrated here:** direct cancellation (same database logic the agent uses): visit from section 6 removed, reserved slots released on the doctor's calendar.

**In Telegram:** the client would request cancellation in natural language; the agent would perform it and confirm in a reply.

**Shown in output:**
1. Success message with date, time, and procedure name.
2. Follow-up list with **no upcoming visits** remaining for this client.

In [ ]:
cancel_service = CancelAppointmentService()
cancel_result = run_async(
    cancel_service.cancel_appointment(
        client_id=demo_client_id,
        date=booking_date,
        time=booking_time,
        procedure_id=int(selected_procedure["procedure_id"]),
    )
)
print(json.dumps(cancel_result, indent=2, default=str))

{
  "success": true,
  "appointment_id": 10,
  "message": "Appointment on 27.06.2026 09:00 (procedure: Botox (forehead)) cancelled successfully"
}


In [ ]:
# Confirm the appointment list is empty (or no longer includes the cancelled visit).
appointments_after_cancel = run_async(
    appointment_service.get_appointment(demo_client_id)
)
print(json.dumps(appointments_after_cancel, indent=2, default=str))

{
  "success": false,
  "error": "No appointments from today onward for client ID 11"
}


In [ ]:
run_async(dispose_engine())